# PHY408 Lab 02: From Collisions to Numbers
### ROOT, the CMS dimuon spectrum and descriptive statistics

**PHY408 Data-Intensive Science** · Xiamen University Malaysia
Week 2, Lesson 2 (hands-on lecture, 2 hours)
**Classroom lesson this week (Lesson 1):** Probability foundations
**Data:** CMS Open Data 2012 DoubleMuParked (NanoAOD-style muon file, streamed); course extract `data/dimuon_extract.npz`; published top-quark mass measurements (PDG compilation)

> **Dimuon thread (Weeks 2-10):** the thread starts here. The extract you build today is the dataset for every lab up to Week 10.

> **Before you start:** work on a *copy* of this notebook inside your personal repository (File → Save As…), so that `git pull` on the course repository never conflicts with your edits.

## The storyline of today

Every part answers one question, and each answer is the input to the next part:

| Part | Question | Tool | Output you keep |
| --- | --- | --- | --- |
| A | How does HEP store and show a distribution? | `TH1F`, `TCanvas`, `TFile` | a histogram saved in a `.root` file |
| B | How do we get a particle's mass from its decay products? | four-vectors (`TLorentzVector`, `PtEtaPhiMVector`) | invariant mass of a toy Z → μμ |
| C | What does real CMS data look like? | `RDataFrame` on CMS Open Data | the dimuon mass spectrum |
| D | How do we avoid rereading terabytes? | `AsNumpy`, `.npz` | **your dimuon extract** |
| E | How do we plot and describe a peak? | Matplotlib, Seaborn, NumPy / SciPy | publication-style figure; Z-peak position and width |
| F | How do we combine many measurements of one number? | pandas, weighted mean, χ² | top-quark mass combination |

The thread in one line: **histogram → four-vector → real events → extract → describe one peak → combine many measurements.**

## Learning objectives

By the end of this lab you should be able to:

1. Create, draw, save and reread ROOT histograms from Python, and explain what the stats box reports
2. Compute the invariant mass of a particle pair and explain why it does not depend on the reference frame
3. Read CMS Open Data with `RDataFrame`, select opposite-sign muon pairs and reconstruct the dimuon spectrum
4. Save a compact analysis extract that later labs reuse
5. Describe a peak with location, spread and shape statistics, and explain why the answers depend on the window you choose
6. Combine independent measurements with an inverse-variance weighted mean and judge their consistency with χ²/ndf

## Lab plan

| Block | Time |
| --- | --- |
| Opening and setup check | 5 min |
| Part A: ROOT histograms and files | 15 min |
| Part B: Four-vectors and invariant mass | 15 min |
| Part C: Real CMS data with RDataFrame | 20 min |
| Part D: Save your dimuon extract | 10 min |
| Part E: Plotting and describing the Z peak | 30 min |
| Part F: Combining measurements: the top-quark mass | 20 min |
| Exit task | 5 min |

## 0. Setup

Run the cell below. If an import fails, check that the `phy408` environment is active (see the course README).

In [ ]:
# --- Setup: run this cell first -------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats
from pathlib import Path

rng = np.random.default_rng(seed=408)      # fixed seed -> reproducible results
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
DATA = Path("..") / "data"                 # shared course data folder

plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True})
print("NumPy", np.__version__, "| pandas", pd.__version__, "| seaborn", sns.__version__)

try:
    import ROOT
    ROOT.gRandom.SetSeed(408)              # reproducible ROOT random numbers too
    print("ROOT", ROOT.gROOT.GetVersion())
except ImportError:
    ROOT = None
    print("PyROOT not found: activate the phy408 environment (see README)")

import sys
sys.path.insert(0, str(Path("..").resolve()))   # course repository root
from common.dimuon import load_dimuon, in_window, PDG_MASS
dimuon = load_dimuon()   # dict of arrays: mass, pt1, pt2, eta1, eta2
print(f"{len(dimuon['mass']):,} dimuon candidates ({dimuon['source']})")

## Part A: ROOT histograms and files (~15 min)

**Question:** how does HEP store and show a distribution?
**Slides:** *What is ROOT?*, *Reading C++*, *TH1F*, *TCanvas*, *TFile I/O*

A `TH1F(name, title, nbins, xmin, xmax)` is a set of bins plus running sums. The title string `"title;x label;y label"` sets the axis labels, and ROOT uses `#mu`, `#sigma` instead of LaTeX backslashes. Values outside `[xmin, xmax)` are **not lost**: they go into the underflow bin (bin 0) and overflow bin (bin `nbins+1`), but they are excluded from the mean and standard deviation shown in the stats box.

In [ ]:
# Part A: worked example (C++ macro on the slide -> PyROOT)
h = ROOT.TH1F("h_gaus", "Gaussian;x;Entries / 0.1", 100, -5, 5)
for x in rng.normal(0, 1, 10_000):
    h.Fill(x)

c = ROOT.TCanvas("cA", "Part A", 700, 450)
h.SetLineColor(ROOT.kBlue + 1); h.SetLineWidth(2)
h.Draw("HIST")
c.Draw()
print(f"Entries = {h.GetEntries():.0f}, Mean = {h.GetMean():.3f}, Std Dev = {h.GetStdDev():.3f}")

In [ ]:
# TODO (Part A):
# 1. Draw 10^4 numbers from N(0, 1) with `rng`, keep them in a NumPy array `xs`,
#    and fill a TH1F with a NARROW range: 40 bins in [-2, 2].
# 2. Compare h.GetMean(), h.GetStdDev() with np.mean(xs), np.std(xs, ddof=1).
#    Print the underflow h.GetBinContent(0) and overflow h.GetBinContent(41).
# 3. Save the histogram to OUT / "lab02_part_a.root" (mode "RECREATE"), close the file,
#    reopen it with ROOT.TFile.Open, Get() the histogram back and print its entries.
xs = ...

**Check your understanding (A):** What do *Entries*, *Mean* and *Std Dev* in the stats box tell you, and why does ROOT's standard deviation come out smaller than NumPy's in your narrow-range histogram?

*Your answer:* 

## Part B: Four-vectors and invariant mass (~15 min)

**Question:** how do we get a particle's mass from its decay products?
**Slides:** *What is a four-vector?*, *Computing invariant mass*, *Collider coordinates*, *Two ways to set a TLorentzVector*

For a parent decaying to two daughters, $M^2 = (E_1+E_2)^2 - |\vec p_1+\vec p_2|^2$ (natural units, everything in GeV). The detector measures $(p_T, \eta, \phi)$, and for muons with $p_T \gg m_\mu$:

$$M^2 \simeq 2\,p_{T1}\,p_{T2}\,\left[\cosh(\eta_1-\eta_2) - \cos(\phi_1-\phi_2)\right].$$

`TLorentzVector` is the classic class you will see in older code and tutorials. Modern ROOT code (and Part C) uses `ROOT.Math.PtEtaPhiMVector` and `ROOT::VecOps::InvariantMass`, which do the same thing faster. Below you see both, so that the jump to Part C is not a surprise.

In [ ]:
# Part B: worked example: a Z at rest decays into two back-to-back muons
import math
M_Z, m_mu = 91.1876, 0.1057                     # GeV (PDG)
p = math.sqrt(M_Z**2 / 4 - m_mu**2)              # two-body decay momentum
E = math.sqrt(p**2 + m_mu**2)

mu1 = ROOT.TLorentzVector(0, 0,  p, E)           # (px, py, pz, E): classic class
mu2 = ROOT.TLorentzVector(0, 0, -p, E)
print(f"TLorentzVector    : M = {(mu1 + mu2).M():.4f} GeV")

# The same in detector coordinates with the modern class (pT, eta, phi, m)
v1 = ROOT.Math.PtEtaPhiMVector(45.0,  1.2,  0.5, m_mu)
v2 = ROOT.Math.PtEtaPhiMVector(40.0, -0.3, -2.4, m_mu)
print(f"PtEtaPhiMVector   : M = {(v1 + v2).M():.4f} GeV")

In [ ]:
# TODO (Part B):
# 1. For the pair v1, v2 above, compute M by hand with the massless formula
#    M^2 = 2 pT1 pT2 [cosh(eta1-eta2) - cos(phi1-phi2)] and compare with (v1+v2).M().
# 2. Frame independence: build mu1, mu2 as TLorentzVectors with SetPtEtaPhiM using the
#    same numbers, then boost BOTH by beta = 0.6 along z (vec.Boost(0, 0, 0.6)) and,
#    separately, along x. Print the pair's M, pT and pz before and after each boost.
#    Which quantities change, and which do not?

**Check your understanding (B):** After the boosts, which of $M$, $p_T$ and $p_z$ stayed the same? Why is that useful at a hadron collider, where the colliding partons carry an unknown fraction of the proton momentum?

*Your answer:* 

## Part C: Real CMS data with RDataFrame (~20 min)

**Question:** what does real CMS data look like?
**Slides:** *RDataFrame: modern declarative analysis*, *Key methods reference*

The file holds one entry per collision event, with variable-length arrays such as `Muon_pt[nMuon]`. RDataFrame builds a chain of `Filter` and `Define` steps that is **lazy**: nothing is read until you ask for a result (`Count().GetValue()`, `Draw`, `AsNumpy`, `Snapshot`).

> We do **not** enable multithreading today: `Range(n)`, which lets you explore on the first *n* events, only works single-threaded.

In [ ]:
# Part C: worked example: open the file and look around
CMS_FILE = ("root://eospublic.cern.ch//eos/opendata/cms/derived-data/"
            "AOD2NanoAODOutreachTool/Run2012BC_DoubleMuParked_Muons.root")

df = ROOT.RDataFrame("Events", CMS_FILE)
print("Columns:", [str(c) for c in df.GetColumnNames()])

df_small = df.Range(200_000)                      # explore on the first 2e5 events
n_mu = df_small.Histo1D(("n_mu", "Muons per event;N_{#mu};Events", 10, -0.5, 9.5), "nMuon")
c = ROOT.TCanvas("cC1", "", 700, 450); n_mu.Draw("HIST"); c.Draw()   # the event loop runs here

In [ ]:
# TODO (Part C):
# 1. Starting from df_small, keep events with exactly two muons of opposite charge
#    (.Filter("nMuon == 2"), then .Filter("Muon_charge[0] != Muon_charge[1]")).
# 2. .Define("Dimuon_mass", "ROOT::VecOps::InvariantMass(Muon_pt, Muon_eta, Muon_phi, Muon_mass)")
# 3. Histogram Dimuon_mass with 300 LOG-spaced bins from 0.25 to 300 GeV
#    (edges = np.logspace(np.log10(0.25), np.log10(300), 301);
#     model = ROOT.RDF.TH1DModel("h_mumu", "title;x;y", 300, edges)),
#    draw it with log-log axes (c.SetLogx(); c.SetLogy()) and save it to outputs/.
# 4. How many events pass each step? Use .Count() on each node. And how many events
#    are in the FULL file (df.Count())?
df_mumu = ...

**Check your understanding (C):** Which resonances can you identify in the spectrum, and at roughly what masses? How many events are in the full file?

*Your answer:* 

## Part D: Save your dimuon extract (~10 min)

**Question:** how do we avoid rereading the full dataset every time?
**Slides:** *RDataFrame key methods* (`Snapshot`), *TFile I/O*

Real analyses run the expensive selection once and keep a small file with only the columns they need. `AsNumpy` turns RDataFrame columns into NumPy arrays; `Snapshot` would write a new ROOT file instead. Our extract uses the same keys as the course copy: `mass, pt1, pt2, eta1, eta2`.

In [ ]:
# TODO (Part D):
# 1. On df_mumu, Define pt1, pt2, eta1, eta2 (e.g. "Muon_pt[0]") and export the five columns
#    with .AsNumpy([...]).
# 2. Save them with np.savez(OUT / "my_dimuon_extract.npz", mass=..., pt1=..., ...).
# 3. Overlay YOUR mass spectrum and the course copy (dimuon["mass"]) on the same log-log
#    Matplotlib plot, both normalised to unit area (density=True). Do they agree in shape?
cols = ...

**Check your understanding (D):** Why do analyses save a small extract instead of rereading the full dataset every time? What do you lose by doing so?

*Your answer:* 

## Part E: Plotting and describing the Z peak (~30 min)

**Question:** how do we show a peak properly, and describe it with numbers before we know how to fit?
**Slides:** *Matplotlib: the fig, ax pattern*, *Histograms and error bars*, *ROOT vs Matplotlib*, *Seaborn*, *Location and spread*, *Skewness and kurtosis*, *Why the window matters*

Plotting rules for this course (they are also the Pset 1 marking rules for figures):

1. Always use the object-oriented API: `fig, ax = plt.subplots()`, then `ax.…`
2. Every axis has a label **with units**; a histogram's y-label states the **bin width** (`Events / 0.5 GeV`)
3. Counted data get error bars (√N); theory curves get lines
4. Save figures as vector PDF with `fig.savefig(...)`

Matplotlib gives you full control of one figure; Seaborn works directly on a pandas DataFrame and is quickest for statistical views (KDE, `hue=` splits, heatmaps, box plots).

From here on we use the **course extract** `dimuon` (loaded in the setup cell), so everyone works with the same numbers.

Two warnings before you start:

* The Z line shape is a Breit–Wigner (Cauchy-like) distribution smeared by detector resolution. A pure Cauchy distribution has **no finite variance**, so the standard deviation you compute depends strongly on how wide a window you take. Robust measures (median, interquartile range, FWHM) are much more stable.
* The window also contains a little non-Z background (Drell–Yan continuum), which pulls the mean.

In [ ]:
# Part E: worked example: the same histogram with ROOT and with Matplotlib
m = np.asarray(dimuon["mass"])
z = m[(m > 70) & (m < 110)]                       # Z window, 70-110 GeV
print(f"{len(z):,} candidates in 70-110 GeV")

# ROOT: Poisson errors come for free
hz = ROOT.TH1F("hz", "Z window;m_{#mu#mu} [GeV];Events / 0.5 GeV", 80, 70, 110)
for x in z:
    hz.Fill(x)
cz = ROOT.TCanvas("cE", "", 700, 450); hz.Draw("E"); cz.Draw()

# Matplotlib: we compute sqrt(N) ourselves
counts, bin_edges = np.histogram(z, bins=80, range=(70, 110))
centres = 0.5 * (bin_edges[1:] + bin_edges[:-1])
fig, ax = plt.subplots()
ax.errorbar(centres, counts, yerr=np.sqrt(counts), fmt="ko", ms=3, capsize=0, label="CMS Open Data 2012")
ax.axvline(91.1876, color="r", ls="--", label="PDG $m_Z$")
ax.set_xlabel(r"$m_{\mu\mu}$ [GeV]"); ax.set_ylabel("Events / 0.5 GeV")
ax.legend(frameon=False); plt.tight_layout()

In [ ]:
# TODO (Part E1, plotting, ~8 min):
# Matplotlib: make ONE figure with two panels, fig, axes = plt.subplots(1, 2, figsize=(11, 4)):
#   left : the Z window 70-110 GeV as a step histogram (histtype="step") with sqrt(N)
#          error bars on top, x-label with units, y-label with the bin width;
#   right: the full spectrum 0.25-300 GeV with log-spaced bins on log-log axes.
#   Add a legend without a frame, call fig.tight_layout(), save to outputs/lab02_z_panels.pdf.
# Seaborn: put pt1 and pt2 of the Z-window pairs into ONE long-form DataFrame with
#   columns "pT" and "muon" (values "leading"/"second"), then
#   sns.histplot(data=..., x="pT", hue="muon", kde=True, element="step", stat="density").
#   Where do the distributions peak? Compare with m_Z / 2.
fig, axes = ...

In [ ]:
# TODO (Part E2, statistics):
# 1. For the 70-110 GeV window compute: N, mean, median, standard deviation (ddof=1),
#    standard error of the mean, interquartile range (np.percentile 75-25),
#    skewness and Pearson kurtosis (stats.kurtosis(..., fisher=False)).
#    Put them in a pandas DataFrame with one column per window.
# 2. Repeat for the windows 60-120 GeV and 81-101 GeV. Which statistics move a lot
#    with the window and which barely move?
# 3. Estimate the FWHM from the 70-110 histogram (`counts`, `centres`): find the
#    maximum, then the outermost bins above half of it.
# 4. Seaborn: for the 70-110 window, build a DataFrame with mass, pt1, pt2, eta1, eta2
#    and draw its correlation matrix with sns.heatmap(..., annot=True, vmin=-1, vmax=1,
#    cmap="coolwarm"). Which pair of variables is most correlated? Why?
def describe(x):
    ...

### Keep your Z-peak estimate (you will need it in Week 4)

Choose **one** number for the peak position and **one** for the width, with a sentence each on why you chose that estimator and window. In Week 4 you will fit the same peak and compare.

In [ ]:
# TODO: fill in your choices and save them
z_estimate = {
    "position_GeV": ...,
    "position_estimator": "...",     # e.g. "median, 81-101 GeV"
    "width_GeV": ...,
    "width_estimator": "...",        # e.g. "FWHM/2.355 from 0.5 GeV bins"
}

**Check your understanding (E):** (1) In your Seaborn plot, the muon $p_T$ distributions pile up just below $m_Z/2 \approx 45.6$ GeV. Why? (2) Why does the standard deviation change so much with the window while the median hardly moves? Is the FWHM you measured the Z width $\Gamma_Z = 2.4955$ GeV? If not, what else contributes?

*Your answer:* 

## Part F: Combining measurements: the top-quark mass (~20 min)

**Question:** Part E described one dataset. How do we combine **many independent measurements** of the same quantity?
**Slides:** *Hands-on: top-quark mass*, Tasks 1–4, *Bonus: weighted average*

Each experiment quotes $m_t \pm \sigma_\text{stat} \pm \sigma_\text{syst}$. With independent uncertainties, the best combination weights each measurement by $w_i = 1/\sigma_i^2$:

$$\hat m = \frac{\sum_i w_i m_i}{\sum_i w_i}, \qquad \sigma_{\hat m} = \frac{1}{\sqrt{\sum_i w_i}}, \qquad \chi^2 = \sum_i w_i (m_i - \hat m)^2 .$$

If $\chi^2/\text{ndf}$ is well above 1 the measurements scatter more than their errors allow. The PDG then inflates the error by the scale factor $S = \sqrt{\chi^2/(N-1)}$ (only when $S > 1$).

> The table below is a teaching compilation of published measurements. Check the values against the current PDG top-quark listing before using them for anything beyond this exercise.

In [ ]:
# Part F: worked example: the measurements and an "ideogram" plot
measurements = {
    "experiment": ["CDF", "CDF", "CDF", "D0", "D0", "D0",
                   "ATLAS", "ATLAS", "CMS", "CMS", "CMS", "CMS", "LHCb"],
    "channel":    ["l+jets", "dilepton", "all-jets", "l+jets", "dilepton", "all-jets",
                   "dilepton", "l+jets", "l+jets", "dilepton", "all-jets", "all-jets", "J/psi"],
    "mass_GeV":   [172.85, 170.28, 172.47, 174.98, 173.32, 171.01,
                   172.08, 172.69, 172.35, 172.22, 172.32, 173.06, 174.00],
    "stat_err":   [0.52, 0.76, 1.43, 0.58, 0.75, 1.16, 0.39, 0.48, 0.14, 0.72, 0.25, 0.84, 0.80],
    "syst_err":   [0.96, 0.86, 1.49, 0.76, 0.89, 1.41, 0.91, 0.88, 0.65, 0.44, 0.39, 0.56, 0.90],
}
top = pd.DataFrame(measurements)
top["total_err"] = np.hypot(top["stat_err"], top["syst_err"])
PDG_MT, PDG_MT_ERR = 172.57, 0.29                  # PDG 2024 direct-measurement average [GeV]

fig, ax = plt.subplots(figsize=(8, 6))
colours = dict(zip(top["experiment"].unique(), sns.color_palette("Set2")))
for i, r in top.iterrows():
    ax.errorbar(r.mass_GeV, i, xerr=r.total_err, fmt="o", capsize=4, color=colours[r.experiment])
ax.axvspan(PDG_MT - PDG_MT_ERR, PDG_MT + PDG_MT_ERR, color="gold", alpha=0.3, label="PDG average")
ax.set_yticks(range(len(top)))
ax.set_yticklabels([f"{r.experiment} ({r.channel})" for _, r in top.iterrows()])
ax.invert_yaxis()
ax.set_xlabel(r"$m_t$ [GeV]"); ax.legend(frameon=False); plt.tight_layout()

In [ ]:
# TODO (Part F):
# 1. Plain statistics of the 13 central values: mean ± standard error, median, std.
# 2. Inverse-variance weighted mean and its error (use total_err).
# 3. chi2, ndf = N - 1, chi2/ndf and the PDG scale factor S (apply only if S > 1).
# 4. Which single measurement carries the largest share of the total weight
#    (w_i / sum(w))? Print the top three.
# 5. (Seaborn, optional) sns.boxplot of mass_GeV by experiment with the PDG value as a
#    horizontal line. What does a box plot of ONE measurement (LHCb) tell you?
w = ...

**Check your understanding (F):** Why do the plain mean and the weighted mean differ? Which measurement dominates, and why? Our weighted error is smaller than the PDG's ±0.29 GeV: what did our simple combination ignore?

*Your answer:* 

## Exit task (~5 min)

1. Make sure every cell runs from top to bottom (Kernel → Restart & Run All).
2. Check that `outputs/` contains: `my_dimuon_extract.npz`, `lab02_dimuon_spectrum.pdf`, `lab02_z_estimate.json`.
3. Commit and push the notebook and `outputs/`. Pset 1 is released this week.
4. Submit the commit SHA (`git log -1 --format=%H`) on Moodle (this step can be done after class).

In [ ]:
# Exit task: commit and push from the terminal:
#   git add lab02_*.ipynb outputs/
#   git commit -m "Lab 02"
#   git push
for f in ["my_dimuon_extract.npz", "lab02_dimuon_spectrum.pdf", "lab02_z_estimate.json"]:
    print(f"{f:28s}", "OK" if (OUT / f).exists() else "MISSING")

## Further reading

- L. Lista, *Statistical Methods for Data Analysis in Particle Physics*, 3rd ed. (2023): descriptive statistics and combination of measurements
- ROOT Primer: https://root.cern/primer/ · PyROOT manual: https://root.cern/manual/python/
- RDataFrame tutorials (see `df102_NanoAODDimuonAnalysis`, which this lab follows): https://root.cern/doc/master/group__tutorial__dataframe.html
- CMS Open Data portal: https://opendata.cern.ch
- PDG review, top quark: https://pdg.lbl.gov
- mplhep (HEP-style Matplotlib): https://github.com/scikit-hep/mplhep